# Part 03: Ice Detection & Phenology Analysis
## Memory-Efficient Chunked Processing Version

This notebook processes ~10M S1 observations across 31,108 lakes using a multi-pass 
chunked architecture to avoid memory overflow on 64GB machines.

**Architecture:**
- **Pass 1:** Build training labels (chunked) → ~1-3M labeled S1 observations
- **Pass 2:** Train RF model and save to disk
- **Pass 3:** Predict on all S1 data (streaming write to CSV)
- **Pass 4:** Detect ice phenology (chunked by spatial chunks)
- **Pass 5:** Upload results to GCS

In [8]:
import pandas as pd
import numpy as np
import geopandas as gpd
from google.cloud import storage
from datetime import datetime, timedelta
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, GroupShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.impute import SimpleImputer
import warnings
import gc
import psutil
import joblib
import os

# ============================================================
# MEMORY MONITORING HELPERS
# ============================================================
def mem(label=""):
    """Print current memory usage."""
    process = psutil.Process()
    mem_gb = process.memory_info().rss / 1e9
    print(f"[MEM{' ' + label if label else ''}] {mem_gb:.1f} GB")
    return mem_gb

def cleanup(*args):
    """Delete objects and run garbage collection."""
    for obj in args:
        try:
            del obj
        except:
            pass
    gc.collect()
    return mem("after cleanup")

# Specific warning suppressions
warnings.filterwarnings('ignore', category=FutureWarning, module='google.api_core')
warnings.filterwarnings('ignore', category=FutureWarning, module='pyproj')
warnings.filterwarnings('ignore', message='.*Shapely GEOS.*')

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Output paths configuration
FIGURES_DIR = './figures/working'
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs('./figures/manuscript', exist_ok=True)

# GCS path for final results
RESULTS_GCS = 'gs://wustl-eeps-geospatial/thermokarst_lakes/results'

print("Libraries loaded successfully")
print(f"Figures directory: {FIGURES_DIR}")
print(f"Results GCS path: {RESULTS_GCS}")
mem("initial")

Libraries loaded successfully
Figures directory: ./figures/working
Results GCS path: gs://wustl-eeps-geospatial/thermokarst_lakes/results
[MEM initial] 0.5 GB


0.52729856

In [9]:
# GCS Export Setup - Install and verify gcsfs
try:
    import gcsfs
    fs = gcsfs.GCSFileSystem()
    print("gcsfs already installed and configured")
except ImportError:
    print("Installing gcsfs...")
    import subprocess
    subprocess.check_call(['pip', 'install', 'gcsfs', '-q'])
    import gcsfs
    fs = gcsfs.GCSFileSystem()
    print("gcsfs installed successfully")

gcsfs already installed and configured


## Configuration

In [ ]:
# ============================================================
# DATA CONFIGURATION
# ============================================================
BUCKET_NAME = 'wustl-eeps-geospatial'
EXPORT_PREFIX = 'thermokarst_lakes/exports'
CHUNKS_PATH = 'thermokarst_lakes/processed/chunks'

YEARS = [2019, 2020, 2021, 2022, 2023]
NUM_CHUNKS = 21  # Chunks 0-20 (spatial chunks)

# Initialize GCS client
storage_client = storage.Client()
bucket = storage_client.bucket(BUCKET_NAME)

# ============================================================
# LABELING THRESHOLDS
# ============================================================
# S2 NDSI thresholds
S2_CLOUD_THRESHOLD = 30  # Maximum cloud percentage to accept
ICE_THRESHOLD_HIGH = 0.8  # s2_ice_fraction > this → ICE label
WATER_THRESHOLD_LOW = 0.2  # s2_ice_fraction < this → WATER label

# Temperature thresholds for sustained periods (training labels)
TEMP_SUSTAINED_COLD = -20  # 10+ consecutive days below → ICE label
TEMP_SUSTAINED_WARM = 10   # 10+ consecutive days above → WATER label
CONSECUTIVE_DAYS = 10

# Temperature conflict thresholds (single-day, for excluding S2)
TEMP_CONFLICT_COLD = -5    # S2 "water" on a day below this → exclude
TEMP_CONFLICT_WARM = 5     # S2 "ice" on a day above this → exclude

# ============================================================
# INFERENCE CONFIGURATION
# ============================================================
# S2 override during inference: Set to False for RF-only classification
# When False, all ice state classifications come from RF model only
# (S2 is still used for TRAINING labels, just not inference overrides)
USE_S2_OVERRIDE = False  # Changed from True to simplify methodology

# ============================================================
# RF MODEL FEATURES
# ============================================================
FEATURE_COLS = [
    'lake_vv_db', 'lake_vh_db', 'lake_vv_vh_ratio',
    'land_vv_db', 'land_vh_db', 'land_vv_vh_ratio'
]

# ============================================================
# PHENOLOGY CONFIGURATION
# ============================================================
MIN_CONSECUTIVE_OBS = 2
ICE_OFF_START_DATE = '04-01'  # April 1 (DOY 91)
ICE_OFF_END_DATE = '08-31'    # August 31 (DOY 243) - prevents overlap with ice-on search
ICE_ON_START_DATE = '09-01'   # September 1 (DOY 244)

# ============================================================
# OUTPUT FILE PATHS
# ============================================================
LOCAL_TIMESERIES_PATH = 'alaska_lakes_ice_probability_timeseries_2019-2023.csv'
LOCAL_PHENOLOGY_PATH = 'alaska_lakes_ice_phenology_2019-2023.csv'
LOCAL_MODEL_PATH = 'rf_ice_model.joblib'
LOCAL_TRAINING_PATH = 'training_data.parquet'

print(f"Configuration loaded:")
print(f"  Years: {YEARS}")
print(f"  Chunks: {NUM_CHUNKS}")
print(f"  Features: {FEATURE_COLS}")
print(f"  USE_S2_OVERRIDE: {USE_S2_OVERRIDE}")

## Data Loading Functions (Chunked)

In [11]:
def load_chunk_data(sensor, year, chunk_id):
    """
    Load a single chunk-year of sensor data from GCS.
    
    Parameters:
    -----------
    sensor : str - 's1', 's2', or 'era5'
    year : int - Year (2019-2023)
    chunk_id : int - Chunk index (0-20)
    
    Returns:
    --------
    pd.DataFrame or None if file doesn't exist
    """
    path = f'gs://{BUCKET_NAME}/{EXPORT_PREFIX}/{year}/chunk_{chunk_id:02d}/{sensor}_data.csv'
    try:
        df = pd.read_csv(path)
        df['year'] = year
        df['chunk'] = chunk_id
        
        # Convert date columns
        date_col = f'{sensor}_date'
        if date_col in df.columns:
            df[date_col] = pd.to_datetime(df[date_col])
        
        return df
    except Exception as e:
        return None

def load_morphometry():
    """Load lake morphometry from chunk files (small, ~31K lakes)."""
    morphometry_chunks = []
    
    for chunk_id in range(NUM_CHUNKS):
        chunk_path = f'gs://{BUCKET_NAME}/{CHUNKS_PATH}/chunk_{chunk_id:02d}.geojson'
        try:
            chunk_gdf = gpd.read_file(chunk_path)
            chunk_gdf['chunk'] = chunk_id
            morphometry_chunks.append(chunk_gdf)
        except Exception as e:
            print(f"  Warning: Could not load chunk {chunk_id:02d}: {e}")
            continue
    
    if len(morphometry_chunks) == 0:
        raise FileNotFoundError("Could not load any chunk files!")
    
    morphometry_gdf = pd.concat(morphometry_chunks, ignore_index=True)
    
    # Rename columns to match expected names
    # Actual columns: id, lake_area_km2, shoreline_dev, circularity, convexity, centroid_lon, centroid_lat
    morphometry_gdf = morphometry_gdf.rename(columns={
        'id': 'lake_id',
        'lake_area_km2': 'area_km2',
        'shoreline_dev': 'sdi'
    })
    
    # Select relevant columns
    morphometry_cols = ['lake_id', 'area_km2', 'circularity', 'sdi', 'convexity', 
                        'centroid_lon', 'centroid_lat', 'chunk']
    morphometry = morphometry_gdf[morphometry_cols].copy()
    
    print(f"Loaded morphometry for {len(morphometry):,} lakes from {len(morphometry_chunks)} chunks")
    return morphometry

print("Data loading functions defined")

Data loading functions defined


## Label Creation Functions

In [12]:
def identify_sustained_temp_labels(era5_df, cold_thresh, warm_thresh, min_consecutive):
    """
    Create training labels from sustained temperature periods.
    
    Returns DataFrame with columns ['lake_id', 'date', 'temp_label']
    """
    temp_labels = []
    
    for lake_id, lake_df in era5_df.groupby('id'):
        lake_df = lake_df.sort_values('era5_date').copy()
        
        # Identify cold periods (< cold_thresh)
        lake_df['is_cold'] = lake_df['temp_c'] < cold_thresh
        lake_df['cold_group'] = (~lake_df['is_cold']).cumsum()
        
        cold_periods = lake_df[lake_df['is_cold']].groupby('cold_group').filter(
            lambda x: len(x) >= min_consecutive
        )
        
        for _, row in cold_periods.iterrows():
            temp_labels.append({
                'lake_id': lake_id,
                'date': row['era5_date'],
                'temp_label': 'ice'
            })
        
        # Identify warm periods (> warm_thresh)
        lake_df['is_warm'] = lake_df['temp_c'] > warm_thresh
        lake_df['warm_group'] = (~lake_df['is_warm']).cumsum()
        
        warm_periods = lake_df[lake_df['is_warm']].groupby('warm_group').filter(
            lambda x: len(x) >= min_consecutive
        )
        
        for _, row in warm_periods.iterrows():
            temp_labels.append({
                'lake_id': lake_id,
                'date': row['era5_date'],
                'temp_label': 'water'
            })
    
    return pd.DataFrame(temp_labels)

def create_s2_labels_with_conflict_detection(s2_df, era5_df, ice_thresh, water_thresh,
                                             conflict_cold, conflict_warm, cloud_thresh):
    """
    Create S2 NDSI labels with temperature conflict detection.
    
    Returns DataFrame with columns ['lake_id', 'date', 's2_label']
    """
    # Filter for low cloud
    s2_clean = s2_df[s2_df['s2_cloud_pct'] <= cloud_thresh].copy()
    
    # Create S2 labels based on NDSI thresholds
    s2_clean['s2_label'] = None
    s2_clean.loc[s2_clean['s2_ice_fraction'] >= ice_thresh, 's2_label'] = 'ice'
    s2_clean.loc[s2_clean['s2_ice_fraction'] <= water_thresh, 's2_label'] = 'water'
    
    # Keep only labeled observations
    s2_labeled = s2_clean[s2_clean['s2_label'].notna()].copy()
    s2_labeled['date'] = s2_labeled['s2_date'].dt.normalize()
    
    # Merge with ERA5 temperature for conflict detection
    era5_for_merge = era5_df[['id', 'era5_date', 'temp_c']].copy()
    era5_for_merge['date'] = era5_for_merge['era5_date'].dt.normalize()
    era5_for_merge = era5_for_merge.rename(columns={'id': 'lake_id'})
    
    s2_with_temp = s2_labeled.merge(
        era5_for_merge[['lake_id', 'date', 'temp_c']],
        left_on=['id', 'date'],
        right_on=['lake_id', 'date'],
        how='left'
    )
    
    # Identify conflicts
    s2_with_temp['conflict'] = False
    # S2 says water but temp is very cold
    s2_with_temp.loc[(s2_with_temp['s2_label'] == 'water') & 
                     (s2_with_temp['temp_c'] < conflict_cold), 'conflict'] = True
    # S2 says ice but temp is warm
    s2_with_temp.loc[(s2_with_temp['s2_label'] == 'ice') & 
                     (s2_with_temp['temp_c'] > conflict_warm), 'conflict'] = True
    
    # Keep only non-conflicting labels
    s2_clean_labels = s2_with_temp[~s2_with_temp['conflict']][['id', 'date', 's2_label']].copy()
    s2_clean_labels = s2_clean_labels.rename(columns={'id': 'lake_id'})
    
    return s2_clean_labels

def create_s1_features(s1_df):
    """Add derived features to S1 data."""
    s1_df = s1_df.copy()
    s1_df['lake_vv_vh_ratio'] = s1_df['lake_vv_db'] - s1_df['lake_vh_db']
    s1_df['land_vv_vh_ratio'] = s1_df['land_vv_db'] - s1_df['land_vh_db']
    s1_df['date'] = s1_df['s1_date'].dt.normalize()
    return s1_df

print("Label creation functions defined")

Label creation functions defined


In [13]:
def detect_ice_phenology(lake_df, min_consecutive=2):
    """
    Detect ice-off and ice-on dates for a single lake.
    
    Parameters:
    -----------
    lake_df : pd.DataFrame - Time series data for one lake, sorted by date
    min_consecutive : int - Minimum consecutive observations to confirm transition
    
    Returns:
    --------
    dict with ice_off/ice_on dates, DOY, confidence, and ice_free_days
    
    Note:
    -----
    Ice-off is only searched for between ICE_OFF_START_DATE (Apr 1) and 
    ICE_OFF_END_DATE (Aug 31) to prevent false detections in fall after
    ice-on has already occurred. This constraint was added to fix ~60 
    anomalous records where ice-off DOY > ice-on DOY.
    """
    lake_df = lake_df.sort_values('date').reset_index(drop=True)
    year = lake_df['date'].iloc[0].year
    
    ice_off_start = pd.Timestamp(f'{year}-{ICE_OFF_START_DATE}')
    ice_off_end = pd.Timestamp(f'{year}-{ICE_OFF_END_DATE}')
    ice_on_start = pd.Timestamp(f'{year}-{ICE_ON_START_DATE}')
    
    result = {
        'ice_off_date': None, 'ice_off_doy': None, 'ice_off_conf': None,
        'ice_on_date': None, 'ice_on_doy': None, 'ice_on_conf': None,
        'ice_free_days': None
    }
    
    # Detect ice-off (first sustained WATER between ice_off_start and ice_off_end)
    # Constrained to Apr 1 - Aug 31 to prevent overlap with ice-on search window
    spring_df = lake_df[(lake_df['date'] >= ice_off_start) & 
                        (lake_df['date'] <= ice_off_end)].copy()
    
    if len(spring_df) >= min_consecutive:
        for i in range(len(spring_df) - min_consecutive + 1):
            window = spring_df.iloc[i:i+min_consecutive]
            if (window['ice_state'] == 'water').all():
                result['ice_off_date'] = window.iloc[0]['date']
                result['ice_off_doy'] = window.iloc[0]['date'].dayofyear
                result['ice_off_conf'] = window.iloc[0]['confidence']
                break
    
    # Detect ice-on (first sustained ICE after ice_on_start)
    fall_df = lake_df[lake_df['date'] >= ice_on_start].copy()
    
    if len(fall_df) >= min_consecutive:
        for i in range(len(fall_df) - min_consecutive + 1):
            window = fall_df.iloc[i:i+min_consecutive]
            if (window['ice_state'] == 'ice').all():
                result['ice_on_date'] = window.iloc[0]['date']
                result['ice_on_doy'] = window.iloc[0]['date'].dayofyear
                result['ice_on_conf'] = window.iloc[0]['confidence']
                break
    
    # Calculate ice-free period
    if result['ice_off_date'] is not None and result['ice_on_date'] is not None:
        result['ice_free_days'] = (result['ice_on_date'] - result['ice_off_date']).days
    
    return result

print("Phenology detection function defined (with ice-off end date constraint)")

Phenology detection function defined (with ice-off end date constraint)


## Load Morphometry Data (Small, ~31K lakes)

In [14]:
# Load morphometry - this is small enough to keep in memory
print("Loading lake morphometry...")
mem("before morphometry")
morphometry = load_morphometry()
mem("after morphometry")

print(f"\nLakes per chunk:")
print(morphometry.groupby('chunk').size())

Loading lake morphometry...
[MEM before morphometry] 0.5 GB
Loaded morphometry for 31,108 lakes from 21 chunks
[MEM after morphometry] 0.7 GB

Lakes per chunk:
chunk
0     1659
1     1402
2     2172
3     1002
4     2243
5      408
6     1726
7     1704
8      753
9      460
10    1440
11    1392
12    1834
13    2001
14    2095
15    1152
16    2019
17     897
18    1493
19    1987
20    1269
dtype: int64


---
# PASS 1: Build Training Labels (Chunked)

Process each chunk-year to create training labels:
1. Load ERA5 → create temperature-based labels
2. Load S2 → create NDSI labels with temperature conflict filtering
3. Load S1 → inner join with labels (only keep labeled S1 observations)
4. Accumulate labeled S1 data (~1-3M rows, not 10M)

In [15]:
# ============================================================
# PASS 1: BUILD TRAINING LABELS (CHUNKED)
# ============================================================
print("="*60)
print("PASS 1: Building training labels from chunks")
print("="*60)
mem("start Pass 1")

# Check if training data already exists
if os.path.exists(LOCAL_TRAINING_PATH):
    print(f"\nLoading existing training data from {LOCAL_TRAINING_PATH}")
    training_data = pd.read_parquet(LOCAL_TRAINING_PATH)
    print(f"Loaded {len(training_data):,} training samples")
    SKIP_PASS_1 = True
else:
    SKIP_PASS_1 = False
    labeled_chunks = []
    total_temp_labels = 0
    total_s2_labels = 0
    
    for chunk_id in range(NUM_CHUNKS):
        chunk_start = mem(f"chunk {chunk_id:02d}")
        chunk_labeled = []
        
        for year in YEARS:
            # Load data for this chunk-year
            era5 = load_chunk_data('era5', year, chunk_id)
            s2 = load_chunk_data('s2', year, chunk_id)
            s1 = load_chunk_data('s1', year, chunk_id)
            
            if era5 is None or s1 is None:
                print(f"  Skipping chunk {chunk_id:02d}/{year}: missing data")
                continue
            
            # Create temperature-based labels
            temp_labels = identify_sustained_temp_labels(
                era5, TEMP_SUSTAINED_COLD, TEMP_SUSTAINED_WARM, CONSECUTIVE_DAYS
            )
            total_temp_labels += len(temp_labels)
            
            # Create S2 labels (if S2 data available)
            if s2 is not None and len(s2) > 0:
                s2_labels = create_s2_labels_with_conflict_detection(
                    s2, era5, ICE_THRESHOLD_HIGH, WATER_THRESHOLD_LOW,
                    TEMP_CONFLICT_COLD, TEMP_CONFLICT_WARM, S2_CLOUD_THRESHOLD
                )
                total_s2_labels += len(s2_labels)
            else:
                s2_labels = pd.DataFrame(columns=['lake_id', 'date', 's2_label'])
            
            # Combine labels (temperature takes precedence)
            if len(temp_labels) > 0:
                temp_labels['label_source'] = 'temperature'
                temp_labels = temp_labels.rename(columns={'temp_label': 'label'})
            
            if len(s2_labels) > 0:
                s2_labels['label_source'] = 's2'
                s2_labels = s2_labels.rename(columns={'s2_label': 'label'})
                
                # Remove S2 labels that conflict with temperature labels
                if len(temp_labels) > 0:
                    temp_keys = set(zip(temp_labels['lake_id'], temp_labels['date']))
                    s2_labels = s2_labels[~s2_labels.apply(
                        lambda x: (x['lake_id'], x['date']) in temp_keys, axis=1
                    )]
            
            # Combine all labels
            all_labels = pd.concat([temp_labels, s2_labels], ignore_index=True)
            
            if len(all_labels) == 0:
                del era5, s2, s1, temp_labels, s2_labels
                gc.collect()
                continue
            
            # Add features to S1 data
            s1 = create_s1_features(s1)
            
            # Inner join S1 with labels (only keep labeled observations)
            s1['lake_id'] = s1['id']
            labeled_s1 = s1.merge(
                all_labels[['lake_id', 'date', 'label', 'label_source']],
                on=['lake_id', 'date'],
                how='inner'
            )
            
            if len(labeled_s1) > 0:
                chunk_labeled.append(labeled_s1)
            
            # Cleanup
            del era5, s2, s1, temp_labels, s2_labels, all_labels, labeled_s1
            gc.collect()
        
        # Append chunk data
        if len(chunk_labeled) > 0:
            chunk_df = pd.concat(chunk_labeled, ignore_index=True)
            labeled_chunks.append(chunk_df)
            print(f"  Chunk {chunk_id:02d}: {len(chunk_df):,} labeled samples")
            del chunk_df
        
        del chunk_labeled
        gc.collect()
        mem(f"end chunk {chunk_id:02d}")
    
    # Combine all labeled data
    print("\nCombining all labeled data...")
    training_data = pd.concat(labeled_chunks, ignore_index=True)
    del labeled_chunks
    gc.collect()
    
    # Create binary label
    training_data['ice_binary'] = (training_data['label'] == 'ice').astype(int)
    
    # Save training data checkpoint
    print(f"\nSaving training data to {LOCAL_TRAINING_PATH}...")
    training_data.to_parquet(LOCAL_TRAINING_PATH, index=False)
    
    print(f"\n{'='*60}")
    print(f"PASS 1 COMPLETE")
    print(f"{'='*60}")
    print(f"Total training samples: {len(training_data):,}")
    print(f"Temperature-based labels: {total_temp_labels:,}")
    print(f"S2-based labels: {total_s2_labels:,}")
    print(f"Label distribution:")
    print(training_data['label'].value_counts())
    print(f"Label source distribution:")
    print(training_data['label_source'].value_counts())

mem("end Pass 1")

PASS 1: Building training labels from chunks
[MEM start Pass 1] 0.7 GB
[MEM chunk 00] 0.7 GB
  Chunk 00: 146,400 labeled samples
[MEM end chunk 00] 0.8 GB
[MEM chunk 01] 0.8 GB
  Chunk 01: 125,334 labeled samples
[MEM end chunk 01] 0.9 GB
[MEM chunk 02] 0.9 GB
  Chunk 02: 296,695 labeled samples
[MEM end chunk 02] 1.1 GB
[MEM chunk 03] 1.1 GB
  Chunk 03: 92,665 labeled samples
[MEM end chunk 03] 1.1 GB
[MEM chunk 04] 1.1 GB
  Chunk 04: 99,615 labeled samples
[MEM end chunk 04] 1.1 GB
[MEM chunk 05] 1.1 GB
  Chunk 05: 70,053 labeled samples
[MEM end chunk 05] 1.1 GB
[MEM chunk 06] 1.1 GB
  Chunk 06: 202,166 labeled samples
[MEM end chunk 06] 1.2 GB
[MEM chunk 07] 1.2 GB
  Chunk 07: 140,101 labeled samples
[MEM end chunk 07] 1.2 GB
[MEM chunk 08] 1.2 GB
  Chunk 08: 128,560 labeled samples
[MEM end chunk 08] 1.2 GB
[MEM chunk 09] 1.2 GB
  Chunk 09: 43,141 labeled samples
[MEM end chunk 09] 1.2 GB
[MEM chunk 10] 1.2 GB
  Chunk 10: 142,250 labeled samples
[MEM end chunk 10] 1.3 GB
[MEM chun

2.500595712

---
# PASS 2: Train Random Forest Model

Train the RF classifier on labeled data and save to disk.

In [16]:
# ============================================================
# PASS 2: TRAIN RF MODEL
# ============================================================
print("="*60)
print("PASS 2: Training Random Forest model")
print("="*60)
mem("start Pass 2")

# Check if model already exists
if os.path.exists(LOCAL_MODEL_PATH):
    print(f"\nLoading existing model from {LOCAL_MODEL_PATH}")
    rf_model = joblib.load(LOCAL_MODEL_PATH)
    print("Model loaded successfully")
    SKIP_TRAINING = True
else:
    SKIP_TRAINING = False
    
    # Sample if needed (keep under 2M for training speed)
    if len(training_data) > 2_000_000:
        print(f"\nSampling from {len(training_data):,} to ~2M samples...")
        training_data = training_data.groupby(['ice_binary', 'label_source'], group_keys=False).apply(
            lambda x: x.sample(min(len(x), 500_000), random_state=42)
        )
        print(f"After sampling: {len(training_data):,} samples")
    
    # Prepare features
    print("\nPreparing features...")
    X = training_data[FEATURE_COLS].values
    y = training_data['ice_binary'].values
    lake_ids = training_data['lake_id'].values
    
    # Handle NaN values
    valid_mask = ~np.isnan(X).any(axis=1)
    X = X[valid_mask]
    y = y[valid_mask]
    lake_ids = lake_ids[valid_mask]
    print(f"After removing NaN: {len(X):,} samples")
    
    # GroupShuffleSplit (same lake never in both train and test)
    print("\nSplitting data...")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=lake_ids))
    
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]
    
    print(f"Training samples: {len(X_train):,}")
    print(f"Test samples: {len(X_test):,}")
    print(f"Training ice fraction: {y_train.mean():.3f}")
    print(f"Test ice fraction: {y_test.mean():.3f}")
    
    # Train
    print("\nTraining Random Forest...")
    rf_model = RandomForestClassifier(
        n_estimators=100,
        max_depth=15,
        min_samples_split=20,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1,
        verbose=1
    )
    rf_model.fit(X_train, y_train)
    
    # Evaluate
    print("\nEvaluating on test set...")
    y_pred = rf_model.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=['water', 'ice']))
    
    # Feature importance
    print("\nFeature Importance:")
    for feat, imp in sorted(zip(FEATURE_COLS, rf_model.feature_importances_), 
                           key=lambda x: -x[1]):
        print(f"  {feat}: {imp:.4f}")
    
    # Save model
    print(f"\nSaving model to {LOCAL_MODEL_PATH}...")
    joblib.dump(rf_model, LOCAL_MODEL_PATH)
    
    # Cleanup training data
    del training_data, X, y, lake_ids, X_train, X_test, y_train, y_test
    gc.collect()

print(f"\n{'='*60}")
print(f"PASS 2 COMPLETE - Model ready")
print(f"{'='*60}")
mem("end Pass 2")

PASS 2: Training Random Forest model
[MEM start Pass 2] 2.5 GB

Sampling from 2,853,532 to ~2M samples...


/var/tmp/ipykernel_16748/2814539671.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  training_data = training_data.groupby(['ice_binary', 'label_source'], group_keys=False).apply(


After sampling: 1,362,958 samples

Preparing features...
After removing NaN: 1,362,958 samples

Splitting data...
Training samples: 1,088,879
Test samples: 274,079
Training ice fraction: 0.651
Test ice fraction: 0.651

Training Random Forest...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  18 tasks      | elapsed:   12.5s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:   42.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.1s



Evaluating on test set...


[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.2s finished


              precision    recall  f1-score   support

       water       0.95      0.96      0.96     95702
         ice       0.98      0.97      0.98    178377

    accuracy                           0.97    274079
   macro avg       0.97      0.97      0.97    274079
weighted avg       0.97      0.97      0.97    274079


Feature Importance:
  land_vh_db: 0.4641
  land_vv_db: 0.2559
  land_vv_vh_ratio: 0.1736
  lake_vv_db: 0.0473
  lake_vh_db: 0.0471
  lake_vv_vh_ratio: 0.0120

Saving model to rf_ice_model.joblib...

PASS 2 COMPLETE - Model ready
[MEM end Pass 2] 2.9 GB


2.907140096

---
# PASS 3: Predict on All S1 Data (Streaming Write)

Process each chunk-year, predict ice probability, and write incrementally to CSV.

In [17]:
# ============================================================
# PASS 3: STREAMING PREDICTIONS
# ============================================================
print("="*60)
print("PASS 3: Predicting on all S1 data (streaming write)")
print("="*60)
mem("start Pass 3")

# Check if timeseries already exists and is complete
if os.path.exists(LOCAL_TIMESERIES_PATH):
    existing_rows = sum(1 for _ in open(LOCAL_TIMESERIES_PATH)) - 1  # minus header
    print(f"Existing timeseries has {existing_rows:,} rows")
    # Expected: ~10M rows, if significantly less, regenerate
    if existing_rows > 9_000_000:
        print("Timeseries appears complete, skipping Pass 3")
        SKIP_PASS_3 = True
    else:
        print("Timeseries incomplete, regenerating...")
        SKIP_PASS_3 = False
else:
    SKIP_PASS_3 = False

if not SKIP_PASS_3:
    # Load model
    rf_model = joblib.load(LOCAL_MODEL_PATH)
    imputer = SimpleImputer(strategy='median')
    
    # Define output columns
    TIMESERIES_COLS = ['lake_id', 's1_date', 'year', 'chunk',
                       'lake_vv_db', 'lake_vh_db', 'lake_vv_vh_ratio',
                       'rf_ice_prob', 'rf_ice_pred']
    
    first_write = True
    total_rows = 0
    
    for chunk_id in range(NUM_CHUNKS):
        chunk_rows = 0
        mem(f"chunk {chunk_id:02d}")
        
        for year in YEARS:
            # Load S1 for this chunk-year
            s1 = load_chunk_data('s1', year, chunk_id)
            
            if s1 is None or len(s1) == 0:
                continue
            
            # Add features
            s1 = create_s1_features(s1)
            s1['lake_id'] = s1['id']
            
            # Prepare features
            X = s1[FEATURE_COLS].values
            if np.isnan(X).any():
                X = imputer.fit_transform(X)
            
            # Predict
            s1['rf_ice_prob'] = rf_model.predict_proba(X)[:, 1]
            s1['rf_ice_pred'] = (s1['rf_ice_prob'] > 0.5).astype(int)
            
            # Select output columns and write
            output_chunk = s1[TIMESERIES_COLS].copy()
            output_chunk.to_csv(
                LOCAL_TIMESERIES_PATH,
                mode='w' if first_write else 'a',
                header=first_write,
                index=False
            )
            first_write = False
            
            chunk_rows += len(output_chunk)
            total_rows += len(output_chunk)
            
            # Cleanup
            del s1, X, output_chunk
            gc.collect()
        
        print(f"  Chunk {chunk_id:02d}: {chunk_rows:,} rows written")
    
    print(f"\n{'='*60}")
    print(f"PASS 3 COMPLETE")
    print(f"{'='*60}")
    print(f"Total rows written: {total_rows:,}")
    print(f"Output file: {LOCAL_TIMESERIES_PATH}")

mem("end Pass 3")

PASS 3: Predicting on all S1 data (streaming write)
[MEM start Pass 3] 2.9 GB
[MEM chunk 00] 2.9 GB


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0

  Chunk 00: 672,266 rows written
[MEM chunk 01] 2.9 GB


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0

  Chunk 01: 537,664 rows written
[MEM chunk 02] 2.9 GB


...

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0

  Chunk 19: 794,238 rows written
[MEM chunk 20] 2.9 GB


[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.0s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.1s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0

  Chunk 20: 465,778 rows written

PASS 3 COMPLETE
Total rows written: 11,222,901
Output file: alaska_lakes_ice_probability_timeseries_2019-2023.csv
[MEM end Pass 3] 2.9 GB


2.91016704

---
# PASS 4: Detect Ice Phenology (Chunked)

Process each spatial chunk to detect ice-on and ice-off dates for all lakes.

In [ ]:
# ============================================================
# PASS 4: PHENOLOGY DETECTION (CHUNKED)
# ============================================================
print("="*60)
print("PASS 4: Detecting ice phenology by chunk")
print("="*60)
print(f"USE_S2_OVERRIDE: {USE_S2_OVERRIDE}")
mem("start Pass 4")

# Load model for re-prediction (faster than reading CSV)
rf_model = joblib.load(LOCAL_MODEL_PATH)
imputer = SimpleImputer(strategy='median')

phenology_results = []
total_lake_years = 0

for chunk_id in range(NUM_CHUNKS):
    mem(f"chunk {chunk_id:02d}")
    
    # Get lakes in this chunk
    chunk_lakes = morphometry[morphometry['chunk'] == chunk_id]['lake_id'].unique()
    
    # Load and predict S1 for all years in this chunk
    chunk_predictions = []
    
    for year in YEARS:
        s1 = load_chunk_data('s1', year, chunk_id)
        
        if s1 is None or len(s1) == 0:
            continue
        
        s1 = create_s1_features(s1)
        s1['lake_id'] = s1['id']
        s1['date'] = s1['s1_date']
        
        # Predict
        X = s1[FEATURE_COLS].values
        if np.isnan(X).any():
            X = imputer.fit_transform(X)
        
        s1['rf_ice_prob'] = rf_model.predict_proba(X)[:, 1]
        s1['ice_state'] = np.where(s1['rf_ice_prob'] > 0.5, 'ice', 'water')
        s1['confidence'] = 'medium'  # RF predictions are medium confidence
        s1['year'] = year
        
        # S2 override logic (only if enabled)
        if USE_S2_OVERRIDE:
            # Load S2 for high-confidence overrides
            s2 = load_chunk_data('s2', year, chunk_id)
            if s2 is not None and len(s2) > 0:
                s2_clean = s2[s2['s2_cloud_pct'] <= S2_CLOUD_THRESHOLD].copy()
                s2_clean['date'] = s2_clean['s2_date'].dt.normalize()
                s2_clean['lake_id'] = s2_clean['id']
                
                # High confidence S2 ice
                s2_ice = s2_clean[s2_clean['s2_ice_fraction'] >= ICE_THRESHOLD_HIGH][['lake_id', 'date']].copy()
                s2_ice['s2_ice'] = True
                
                # High confidence S2 water
                s2_water = s2_clean[s2_clean['s2_ice_fraction'] <= WATER_THRESHOLD_LOW][['lake_id', 'date']].copy()
                s2_water['s2_water'] = True
                
                # Merge S2 overrides
                s1 = s1.merge(s2_ice, on=['lake_id', 'date'], how='left')
                s1 = s1.merge(s2_water, on=['lake_id', 'date'], how='left')
                
                # Apply S2 overrides
                s1.loc[s1['s2_ice'] == True, 'ice_state'] = 'ice'
                s1.loc[s1['s2_ice'] == True, 'confidence'] = 'high'
                s1.loc[s1['s2_water'] == True, 'ice_state'] = 'water'
                s1.loc[s1['s2_water'] == True, 'confidence'] = 'high'
                
                del s2, s2_clean, s2_ice, s2_water
        
        chunk_predictions.append(s1[['lake_id', 'date', 'year', 'ice_state', 'confidence']])
        del s1, X
        gc.collect()
    
    if len(chunk_predictions) == 0:
        print(f"  Chunk {chunk_id:02d}: no data")
        continue
    
    chunk_combined = pd.concat(chunk_predictions, ignore_index=True)
    del chunk_predictions
    gc.collect()
    
    # Detect phenology for each lake-year in this chunk
    chunk_lake_years = 0
    for lake_id in chunk_lakes:
        lake_data = chunk_combined[chunk_combined['lake_id'] == lake_id]
        
        for year in YEARS:
            lake_year = lake_data[lake_data['year'] == year].copy()
            
            if len(lake_year) < MIN_CONSECUTIVE_OBS:
                continue
            
            result = detect_ice_phenology(lake_year, min_consecutive=MIN_CONSECUTIVE_OBS)
            result['lake_id'] = lake_id
            result['year'] = year
            phenology_results.append(result)
            chunk_lake_years += 1
    
    del chunk_combined, lake_data
    gc.collect()
    
    print(f"  Chunk {chunk_id:02d}: {len(chunk_lakes)} lakes, {chunk_lake_years} lake-years processed")
    total_lake_years += chunk_lake_years

# Create phenology DataFrame
print("\nCreating phenology DataFrame...")
phenology_df = pd.DataFrame(phenology_results)

# Merge with morphometry
final_results = phenology_df.merge(morphometry, on='lake_id', how='left')

# Reorder columns
col_order = ['lake_id', 'year', 'centroid_lon', 'centroid_lat', 'area_km2', 
             'circularity', 'sdi', 'convexity', 'chunk',
             'ice_off_date', 'ice_off_doy', 'ice_off_conf',
             'ice_on_date', 'ice_on_doy', 'ice_on_conf', 'ice_free_days']
final_results = final_results[[c for c in col_order if c in final_results.columns]]

# Save locally
print(f"\nSaving phenology to {LOCAL_PHENOLOGY_PATH}...")
final_results.to_csv(LOCAL_PHENOLOGY_PATH, index=False)

print(f"\n{'='*60}")
print(f"PASS 4 COMPLETE")
print(f"{'='*60}")
print(f"Total lake-years processed: {total_lake_years:,}")
print(f"Unique lakes: {final_results['lake_id'].nunique():,}")
print(f"Ice-off detected: {final_results['ice_off_date'].notna().sum():,}")
print(f"Ice-on detected: {final_results['ice_on_date'].notna().sum():,}")
print(f"Both detected: {(final_results['ice_off_date'].notna() & final_results['ice_on_date'].notna()).sum():,}")

# Verify confidence distribution
print(f"\nConfidence distribution:")
print(f"  Ice-off high: {(final_results['ice_off_conf'] == 'high').sum():,}")
print(f"  Ice-off medium: {(final_results['ice_off_conf'] == 'medium').sum():,}")
print(f"  Ice-on high: {(final_results['ice_on_conf'] == 'high').sum():,}")
print(f"  Ice-on medium: {(final_results['ice_on_conf'] == 'medium').sum():,}")

mem("end Pass 4")

---
# PASS 5: Upload Results to GCS

In [ ]:
# ============================================================
# PASS 5: UPLOAD TO GCS
# ============================================================
print("="*60)
print("PASS 5: Uploading results to GCS")
print("="*60)

import subprocess

def upload_to_gcs(local_path, gcs_path):
    """Upload file to GCS and verify."""
    print(f"\nUploading {local_path}...")
    result = subprocess.run(
        ['gsutil', 'cp', local_path, gcs_path],
        capture_output=True, text=True
    )
    if result.returncode == 0:
        print(f"  SUCCESS: {gcs_path}")
        # Verify
        ls_result = subprocess.run(
            ['gsutil', 'ls', '-l', gcs_path],
            capture_output=True, text=True
        )
        print(f"  {ls_result.stdout.strip()}")
    else:
        print(f"  FAILED: {result.stderr}")

# Upload timeseries
upload_to_gcs(
    LOCAL_TIMESERIES_PATH,
    f'{RESULTS_GCS}/{LOCAL_TIMESERIES_PATH}'
)

# Upload phenology
upload_to_gcs(
    LOCAL_PHENOLOGY_PATH,
    f'{RESULTS_GCS}/{LOCAL_PHENOLOGY_PATH}'
)

# Upload model
upload_to_gcs(
    LOCAL_MODEL_PATH,
    f'{RESULTS_GCS}/{LOCAL_MODEL_PATH}'
)

print(f"\n{'='*60}")
print(f"PASS 5 COMPLETE")
print(f"{'='*60}")

# List all files in results directory
print("\nAll files in results directory:")
subprocess.run(['gsutil', 'ls', '-lh', RESULTS_GCS])

---
# Summary Statistics and Validation

In [ ]:
# ============================================================
# SUMMARY STATISTICS
# ============================================================
print("="*60)
print("FINAL SUMMARY")
print("="*60)

# Load results if not in memory
if 'final_results' not in dir():
    final_results = pd.read_csv(LOCAL_PHENOLOGY_PATH)
    final_results['ice_off_date'] = pd.to_datetime(final_results['ice_off_date'])
    final_results['ice_on_date'] = pd.to_datetime(final_results['ice_on_date'])

# Complete records (both ice-off and ice-on detected)
complete_records = final_results[
    final_results['ice_off_date'].notna() & 
    final_results['ice_on_date'].notna()
].copy()

print(f"\nDataset Summary:")
print(f"  Total lake-year records: {len(final_results):,}")
print(f"  Unique lakes: {final_results['lake_id'].nunique():,}")
print(f"  Years: {sorted(final_results['year'].unique())}")

print(f"\nDetection Success:")
print(f"  Ice-off detected: {final_results['ice_off_date'].notna().sum():,} ({100*final_results['ice_off_date'].notna().mean():.1f}%)")
print(f"  Ice-on detected: {final_results['ice_on_date'].notna().sum():,} ({100*final_results['ice_on_date'].notna().mean():.1f}%)")
print(f"  Both detected: {len(complete_records):,} ({100*len(complete_records)/len(final_results):.1f}%)")

print(f"\nPhenology Statistics (complete records only):")
print(f"  Ice-off DOY: mean={complete_records['ice_off_doy'].mean():.1f}, median={complete_records['ice_off_doy'].median():.0f}")
print(f"  Ice-on DOY: mean={complete_records['ice_on_doy'].mean():.1f}, median={complete_records['ice_on_doy'].median():.0f}")
print(f"  Ice-free days: mean={complete_records['ice_free_days'].mean():.1f}, median={complete_records['ice_free_days'].median():.0f}")

print(f"\nIce-free Period Statistics:")
print(f"  Min: {complete_records['ice_free_days'].min():.0f} days")
print(f"  Max: {complete_records['ice_free_days'].max():.0f} days")
print(f"  Std: {complete_records['ice_free_days'].std():.1f} days")

In [ ]:
# ============================================================
# VALIDATION FIGURES
# ============================================================

# Figure 1: Phenology distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Ice-off DOY histogram
ax = axes[0, 0]
ax.hist(complete_records['ice_off_doy'], bins=30, edgecolor='black', alpha=0.7)
ax.axvline(complete_records['ice_off_doy'].median(), color='red', linestyle='--', linewidth=2,
           label=f'Median: {complete_records["ice_off_doy"].median():.0f}')
ax.set_xlabel('Day of Year')
ax.set_ylabel('Count')
ax.set_title('Ice-off Date Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Ice-on DOY histogram
ax = axes[0, 1]
ax.hist(complete_records['ice_on_doy'], bins=30, edgecolor='black', alpha=0.7, color='orange')
ax.axvline(complete_records['ice_on_doy'].median(), color='red', linestyle='--', linewidth=2,
           label=f'Median: {complete_records["ice_on_doy"].median():.0f}')
ax.set_xlabel('Day of Year')
ax.set_ylabel('Count')
ax.set_title('Ice-on Date Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Ice-free period histogram
ax = axes[1, 0]
ax.hist(complete_records['ice_free_days'], bins=30, edgecolor='black', alpha=0.7, color='green')
ax.axvline(complete_records['ice_free_days'].median(), color='red', linestyle='--', linewidth=2,
           label=f'Median: {complete_records["ice_free_days"].median():.0f}')
ax.set_xlabel('Days')
ax.set_ylabel('Count')
ax.set_title('Ice-free Period Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

# Ice-free vs lake size
ax = axes[1, 1]
ax.scatter(complete_records['area_km2'], complete_records['ice_free_days'], 
           alpha=0.3, s=10)
ax.set_xlabel('Lake Area (km²)')
ax.set_ylabel('Ice-free Period (days)')
ax.set_title('Ice-free Duration vs Lake Size')
ax.set_xscale('log')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/phenology_distributions.png', dpi=150, bbox_inches='tight')
plt.savefig('./figures/manuscript/fig02_phenology_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: phenology_distributions.png")

In [ ]:
# Figure 2: Latitude relationships
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
from scipy import stats

# Ice-off vs latitude
ax = axes[0]
ax.scatter(complete_records['centroid_lat'], complete_records['ice_off_doy'], alpha=0.3, s=10)
mask = complete_records['centroid_lat'].notna() & complete_records['ice_off_doy'].notna()
slope, intercept, r, p, se = stats.linregress(
    complete_records.loc[mask, 'centroid_lat'], 
    complete_records.loc[mask, 'ice_off_doy']
)
x_line = np.linspace(complete_records['centroid_lat'].min(), complete_records['centroid_lat'].max(), 100)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'r={r:.3f}, p={p:.2e}')
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Ice-off DOY')
ax.set_title('Ice-off Date vs Latitude')
ax.legend()
ax.grid(True, alpha=0.3)

# Ice-on vs latitude
ax = axes[1]
ax.scatter(complete_records['centroid_lat'], complete_records['ice_on_doy'], alpha=0.3, s=10, color='orange')
mask = complete_records['centroid_lat'].notna() & complete_records['ice_on_doy'].notna()
slope, intercept, r, p, se = stats.linregress(
    complete_records.loc[mask, 'centroid_lat'], 
    complete_records.loc[mask, 'ice_on_doy']
)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'r={r:.3f}, p={p:.2e}')
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Ice-on DOY')
ax.set_title('Ice-on Date vs Latitude')
ax.legend()
ax.grid(True, alpha=0.3)

# Ice-free days vs latitude
ax = axes[2]
ax.scatter(complete_records['centroid_lat'], complete_records['ice_free_days'], alpha=0.3, s=10, color='green')
mask = complete_records['centroid_lat'].notna() & complete_records['ice_free_days'].notna()
slope, intercept, r, p, se = stats.linregress(
    complete_records.loc[mask, 'centroid_lat'], 
    complete_records.loc[mask, 'ice_free_days']
)
ax.plot(x_line, slope * x_line + intercept, 'r-', linewidth=2, label=f'r={r:.3f}, p={p:.2e}')
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Ice-free Period (days)')
ax.set_title('Ice-free Duration vs Latitude')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/latitude_phenology_correlations.png', dpi=150, bbox_inches='tight')
plt.savefig('./figures/manuscript/fig05_latitude_phenology.png', dpi=150, bbox_inches='tight')
plt.show()

print("Saved: latitude_phenology_correlations.png")
print(f"\nLatitude correlations:")
print(f"  Ice-off DOY: r = {r:.3f}")

In [ ]:
# ============================================================
# FIGURE 3: Example Lake Time Series (6-panel, diverse sample)
# Shows: rf_ice_prob (continuous), ERA5 temp (secondary y-axis),
#        S2 ice fraction (green dots), ice-off (red), ice-on (blue)
# ============================================================
# 
# Lakes pre-selected for good S2 coverage (940-1207 observations each)
# while maintaining diversity in size and shape classes.
# Selection criteria:
#   - Lakes from high-S2-coverage chunks (2, 3, 5, 6, 7, 8, 15, 19)
#   - Each lake has >= 20 cloud-free S2 observations
#   - 2 lakes per size class (Small, Medium, Large)
#   - Shape variety (Circular, Moderate) within each size class
# ============================================================

def get_size_class(area_km2):
    """Classify lake by size."""
    if area_km2 < 0.1:
        return 'Small'
    elif area_km2 < 0.5:
        return 'Medium'
    else:
        return 'Large'

def get_shape_class(sdi):
    """Classify lake shape by shoreline development index."""
    if pd.isna(sdi):
        return 'Unknown'
    elif sdi < 1.5:
        return 'Circular'
    elif sdi < 2.0:
        return 'Moderate'
    else:
        return 'Complex'

# Pre-selected lakes with excellent S2 coverage (sorted by area, smallest to largest)
# Lake ID | Size Class | Shape Class | Area (km²) | SDI  | S2 Obs | Chunk
# 635251  | Small      | Circular    | 0.034      | 1.42 | 1167   | 2
# 635426  | Small      | Moderate    | 0.058      | 1.57 | 1207   | 15
# 506280  | Medium     | Moderate    | 0.104      | 1.64 | 1206   | 6
# 307670  | Medium     | Circular    | 0.127      | 1.40 | 1152   | 3
# 636107  | Large      | Circular    | 0.640      | 1.35 | 940    | 2
# 507900  | Large      | Moderate    | 2.911      | 1.67 | 1186   | 6

SELECTED_LAKES = [635251, 635426, 506280, 307670, 636107, 507900]
SELECTED_LAKES_S2_COUNTS = {635251: 1167, 635426: 1207, 506280: 1206, 307670: 1152, 636107: 940, 507900: 1186}

print("Using pre-selected lakes with good S2 coverage for time series figure...")
print(f"Selected {len(SELECTED_LAKES)} lakes with {min(SELECTED_LAKES_S2_COUNTS.values())}-{max(SELECTED_LAKES_S2_COUNTS.values())} S2 observations each")

# Get complete records with good phenology detection
complete_with_morph = complete_records.merge(
    morphometry[['lake_id', 'sdi']], on='lake_id', how='left', suffixes=('', '_morph')
)

# Classify lakes
complete_with_morph['size_class'] = complete_with_morph['area_km2'].apply(get_size_class)
complete_with_morph['shape_class'] = complete_with_morph['sdi'].apply(get_shape_class)

# Filter to selected lakes and sort by area
lake_info_df = complete_with_morph[complete_with_morph['lake_id'].isin(SELECTED_LAKES)].drop_duplicates('lake_id')
lake_info_df = lake_info_df.sort_values('area_km2')
selected_lakes = lake_info_df['lake_id'].values

print(f"\nLake details:")
for lid in selected_lakes:
    info = lake_info_df[lake_info_df['lake_id'] == lid].iloc[0]
    s2_count = SELECTED_LAKES_S2_COUNTS.get(lid, 0)
    print(f"  Lake {lid}: {info['area_km2']:.3f} km² ({info['size_class']}), "
          f"SDI={info['sdi']:.2f} ({info['shape_class']}), {s2_count} S2 obs")

# Create figure (3 rows x 2 columns = 6 panels)
fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

# Process each lake
for idx, lake_id in enumerate(selected_lakes):
    if idx >= 6:
        break
        
    ax = axes[idx]
    lake_info = lake_info_df[lake_info_df['lake_id'] == lake_id].iloc[0]
    lake_chunk = morphometry[morphometry['lake_id'] == lake_id]['chunk'].values[0]
    
    # Collect data across all years for this lake
    all_s1 = []
    all_era5 = []
    all_s2 = []
    
    for year in YEARS:
        # Load S1 with predictions
        s1 = load_chunk_data('s1', year, lake_chunk)
        if s1 is not None:
            s1 = create_s1_features(s1)
            s1['lake_id'] = s1['id']
            lake_s1 = s1[s1['lake_id'] == lake_id].copy()
            if len(lake_s1) > 0:
                # Predict
                X = lake_s1[FEATURE_COLS].values
                if np.isnan(X).any():
                    X = imputer.fit_transform(X)
                lake_s1['rf_ice_prob'] = rf_model.predict_proba(X)[:, 1]
                all_s1.append(lake_s1[['s1_date', 'rf_ice_prob', 'lake_vv_db']])
        
        # Load ERA5
        era5 = load_chunk_data('era5', year, lake_chunk)
        if era5 is not None:
            lake_era5 = era5[era5['id'] == lake_id].copy()
            if len(lake_era5) > 0:
                all_era5.append(lake_era5[['era5_date', 'temp_c']])
        
        # Load S2
        s2 = load_chunk_data('s2', year, lake_chunk)
        if s2 is not None:
            lake_s2 = s2[(s2['id'] == lake_id) & (s2['s2_cloud_pct'] <= 30)].copy()
            if len(lake_s2) > 0:
                all_s2.append(lake_s2[['s2_date', 's2_ice_fraction']])
    
    # Combine data
    if all_s1:
        s1_combined = pd.concat(all_s1, ignore_index=True)
        s1_combined = s1_combined.sort_values('s1_date')
    else:
        continue
    
    if all_era5:
        era5_combined = pd.concat(all_era5, ignore_index=True)
        era5_combined = era5_combined.sort_values('era5_date')
    else:
        era5_combined = pd.DataFrame()
    
    if all_s2:
        s2_combined = pd.concat(all_s2, ignore_index=True)
        s2_combined = s2_combined.sort_values('s2_date')
    else:
        s2_combined = pd.DataFrame()
    
    # Plot ice probability (primary y-axis)
    ax.plot(s1_combined['s1_date'], s1_combined['rf_ice_prob'], 
            '-', color='steelblue', linewidth=1.5, alpha=0.8, label='P(ice)')
    
    # Plot S2 ice fraction (green dots)
    s2_count = len(s2_combined)
    if s2_count > 0:
        ax.scatter(s2_combined['s2_date'], s2_combined['s2_ice_fraction'],
                   s=40, marker='o', color='green', alpha=0.6, label=f'S2 ice frac (n={s2_count})', zorder=5)
    
    # Get ALL phenology dates for this lake (all years)
    lake_pheno_all = complete_records[complete_records['lake_id'] == lake_id]
    
    # Plot ice-off (red dashed) and ice-on (blue dashed) for ALL years
    first_ice_off = True
    first_ice_on = True
    for _, lake_pheno in lake_pheno_all.iterrows():
        if pd.notna(lake_pheno['ice_off_date']):
            ax.axvline(lake_pheno['ice_off_date'], color='red', linestyle='--', 
                       linewidth=1.5, alpha=0.7, label='Ice-off' if first_ice_off else None)
            first_ice_off = False
        
        if pd.notna(lake_pheno['ice_on_date']):
            ax.axvline(lake_pheno['ice_on_date'], color='blue', linestyle='--', 
                       linewidth=1.5, alpha=0.7, label='Ice-on' if first_ice_on else None)
            first_ice_on = False
    
    # Secondary y-axis for temperature
    ax2 = ax.twinx()
    if len(era5_combined) > 0:
        ax2.plot(era5_combined['era5_date'], era5_combined['temp_c'],
                 '-', color='orange', linewidth=1, alpha=0.6, label='Temperature')
        ax2.axhline(0, color='orange', linestyle=':', linewidth=1, alpha=0.5)
        ax2.set_ylabel('T (°C)', color='orange', fontsize=10)
        ax2.tick_params(axis='y', labelcolor='orange')
        ax2.set_ylim(-35, 25)
    
    # Configure primary axis
    ax.set_ylim(-0.05, 1.05)
    ax.set_ylabel('P(ice)', fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Title with lake info
    title = f"Lake {lake_id}\n{lake_info['area_km2']:.3f} km² ({lake_info['size_class']}), SDI={lake_info['sdi']:.2f} ({lake_info['shape_class']})"
    ax.set_title(title, fontsize=11, fontweight='bold')
    
    # Legend for first panel only
    if idx == 0:
        lines1, labels1 = ax.get_legend_handles_labels()
        lines2, labels2 = ax2.get_legend_handles_labels()
        ax.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=9, ncol=2)

# Cleanup empty panels
for idx in range(len(selected_lakes), 6):
    axes[idx].set_visible(False)

plt.suptitle('Ice Probability Time Series: Diverse Lake Sample (2019-2023)\nOrdered by lake size (smallest to largest)', 
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/example_lake_timeseries.png', dpi=150, bbox_inches='tight')
plt.savefig('./figures/manuscript/fig03_example_timeseries.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nSaved: figures/working/example_lake_timeseries.png")
print("Saved: figures/manuscript/fig03_example_timeseries.png")

In [ ]:
# Final memory status
print("="*60)
print("PROCESSING COMPLETE")
print("="*60)
mem("final")

print("\nOutput files:")
print(f"  1. {LOCAL_TIMESERIES_PATH} - Ice probability for all S1 observations")
print(f"  2. {LOCAL_PHENOLOGY_PATH} - Ice phenology for all lake-years")
print(f"  3. {LOCAL_MODEL_PATH} - Trained RF model")
print(f"\nGCS location: {RESULTS_GCS}/")